# 👀 Multilayer perceptron (MLP)

In this notebook, we'll walk through the steps required to train your own multilayer perceptron on the CIFAR dataset

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import os
os.environ["KERAS_BACKEND"] = "torch"  # Keras only loads the dataset; this keeps TensorFlow out of the PyTorch process
from keras import datasets

from notebooks.utils_pytorch import display

In [ ]:
# Check GPU - PyTorch needs to be told which device to use
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
NUM_CLASSES = 10

## 1. Prepare the Data <a name="prepare"></a>

In [ ]:
# Keras downloads the dataset once (cached in ~/.keras/datasets) and returns NumPy
# arrays: images of shape (N, 32, 32, 3) uint8 and labels of shape (N, 1)
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# Flatten the labels to shape (N,)
(y_train, y_test) = (y_train.flatten(), y_test.flatten())

In [ ]:
# Scale to [0, 1] and move channels first: (N, H, W, C) -> (N, C, H, W),
# which is the image layout PyTorch layers expect
x_train = torch.tensor(x_train, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
x_test = torch.tensor(x_test, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0

# No one-hot encoding needed: nn.CrossEntropyLoss takes integer class ids
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
display(x_train[:10])
print(y_train[:10])

## 2. Build the model <a name="build"></a>

In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(32 * 32 * 3, 200),
    nn.ReLU(),
    nn.Linear(200, 150),
    nn.ReLU(),
    # No softmax here: the layer outputs raw scores (logits) and
    # nn.CrossEntropyLoss applies the softmax internally
    nn.Linear(150, NUM_CLASSES),
).to(device)

print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Train the model <a name="train"></a>

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=0.0005)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)

for epoch in range(10):
    model.train()
    total_loss, correct = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)  # forward pass
        loss = loss_fn(logits, y)

        opt.zero_grad()  # reset gradients from the previous step
        loss.backward()  # backpropagation
        opt.step()  # update the weights

        total_loss += loss.item() * len(x)
        correct += (logits.argmax(dim=1) == y).sum().item()

    n = len(train_loader.dataset)
    print(f"Epoch {epoch + 1}/10 - loss: {total_loss / n:.4f} - accuracy: {correct / n:.4f}")

## 4. Evaluation <a name="evaluate"></a>

In [ ]:
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=32)

model.eval()
total_loss, correct = 0.0, 0
with torch.no_grad():  # no gradients needed for evaluation
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(x)
        correct += (logits.argmax(dim=1) == y).sum().item()

n = len(test_loader.dataset)
print(f"loss: {total_loss / n:.4f} - accuracy: {correct / n:.4f}")

In [ ]:
CLASSES = np.array(
    [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]
)

model.eval()
with torch.no_grad():
    preds = torch.cat([model(x.to(device)).softmax(dim=1).cpu() for x, _ in test_loader])

preds_single = CLASSES[preds.argmax(dim=1).numpy()]
actual_single = CLASSES[y_test.numpy()]

In [ ]:
n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx].permute(1, 2, 0)  # (C, H, W) -> (H, W, C) for matplotlib
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        "pred = " + str(preds_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        -0.7,
        "act = " + str(actual_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(img)